# Two-Layer Teacher Temperature LR Sweep

We define an energy function $T: \mathbb{R}^{d} \to \mathbb{R}^{k}$, where $T$ takes the form
$$
T(x) = \frac{1}{\gamma n^{a_{2}}}W^{2} g\left( \frac{1}{d^{a_{1}}}W^{1} x \right),
$$
with (1) $W^2 = G\sqrt{ \Sigma }$ for $G \sim \mathcal{N}(0, n^{-b_{2}} I_{k \times n})$ and $\Sigma \in \mathbb{R}^{n \times n}$ diagonal and deterministic with $\Sigma_{ii} = i^{-2\alpha}$ for some $\alpha > 0$; (2) $W^{1} \sim \mathcal{N}(0, d^{-b_{1}}I_{n \times d})$; and (3) $g \in C^{1}(\mathbb{R})$ acting elementwise. We refer to this as a random sample from the ensemble $(d, n, k, \Sigma)$.

Define $h^{1} =  \frac{1}{\sqrt{ d }}W^{1} x$ and $h^{2} = \gamma T(x)$.

We choose the exponents in the teacher in order to (1) keep the RMS norms of $h^{1}$ and $h^{2}$ order-1 at initialization and to (2) also keep the feature map $\frac{\partial T}{\partial h^{1}}$ with RMS norm $\Theta(1)$. Indeed, for $\alpha \neq \frac{1}{2}$ the space of allowable scalings is
$$
b_{1}=1-2a_{1},\qquad b_{2}=s_{\alpha}-2a_{2}-2c,
$$
where $\gamma = n^{c}$ for $c \geq 0$ and 

$$
s_{\alpha} = \log_{n}\sum_{i = 1}^{n} n^{-2\alpha} \sim 
\begin{cases}
1 - 2 \alpha & \alpha < \frac{1}{2} \\
0 & \alpha > \frac{1}{2}.
\end{cases}
$$
We provisionally consider the easy regime $\alpha > \frac{1}{2}$, and make a choice of scalings that doesn't fully absorb the power exponent of $\gamma$ into $a_{2}$:
$$
\gamma=n^{c},\qquad a_{1}=\frac{1}{2},\qquad b_{1}=0,\qquad a_{2}=\frac{s_{\alpha}-2c}{4} = -\frac{c}{2},\qquad b_{2}=\frac{s_{\alpha}-2c}{2} = -c.
$$

We consider the following experimental procedure:

1. Randomly draw a teacher from the ensemble $(d, n, k, \Sigma)$ and a student from the ensemble $(d, \kappa n, k, I_{n})$.
2. Draw an iid train set $\mathcal{S} := \{(x_i, y_i) : i \in [p]\}$ and an independent iid test set $\mathcal{T} := \{(x'_i, y'_i) : i \in [p]\}$ with $x_i, x'_i \sim \mathcal{N}(0, I_d)$. Sample labels from the $\beta$-inverse temperature Gibbs distribution
$$
P(y = j |x) \propto \exp(\beta T_j(x)).
$$
We use the Gumbel-max trick as necessary to simulate sampling. Every recorded observable that depends on sampled inputs is evaluated separately on train and test.
3. Train the student network using $T$ steps of SGD on the train set. For minibatch SGD, the batch size is $\operatorname{round}(p^{\delta})$, unless an explicit `minibatch_size` override is supplied.

We sweep this experimental procedure over $\beta$ in log space, from $\beta = 10^{-3}$ to $10^{2}$.

First, let's find the optimal learning rate for each $\beta$ when using the SGD optimizer. Choose $d = k = 100$, $n = 3d = 300$, $\alpha = \frac{3}{4}$, $p = 4n = 1200$, and $\delta = \frac{1}{2}$.

We plot a Pareto curve for each $\beta$: learning rate on the x-axis, test loss after training on the y-axis. We make one plot for minibatch SGD and one for population gradient descent.


In [ ]:
import sys
from pathlib import Path

# Add parent directory to path to import two_layer_experiment
sys.path.insert(0, str(Path('.').resolve().parent))

import matplotlib.pyplot as plt
import pandas as pd

from two_layer_experiment.results import best_learning_rates, load_results
from two_layer_experiment.plotting import pareto_plot

runs_dir = Path('../runs/two_layer_lr_sweep')
df = load_results(runs_dir)
best = best_learning_rates(df, metric='test_xent')
modes = list(df['mode'].drop_duplicates())
best


In [ ]:
for mode in modes:
    fig, ax = plt.subplots(figsize=(9, 5.5))
    pareto_plot(df, mode, ax=ax, metric='test_xent')
    fig.tight_layout()
    fig.savefig(runs_dir / f'pareto_{mode}_test_xent.png', dpi=180)
    plt.show()


In [ ]:
def best_lr_timeseries(df, best, mode):
    rows = []
    for _, row in best[best['mode'] == mode].iterrows():
        rows.append(df[(df['mode'] == mode) & (df['beta'] == row['beta']) & (df['lr'] == row['lr'])])
    return pd.concat(rows, ignore_index=True)

def plot_metric_over_time(ts, metric, mode):
    fig, ax = plt.subplots(figsize=(9, 5.2))
    for beta, group in ts.groupby('beta', sort=True):
        group = group.sort_values('step')
        ax.plot(group['step'], group[metric], marker='o', linewidth=1.5, label=f'beta={beta:g}')
    ax.set_title(f'{mode}: {metric} at best final test-xent LR')
    ax.set_xlabel('SGD step')
    ax.set_ylabel(metric)
    ax.grid(True, alpha=0.25)
    ax.legend(ncols=2, fontsize='small')
    fig.tight_layout()
    fig.savefig(runs_dir / f'time_{mode}_{metric}.png', dpi=180)
    return fig

def plot_student_teacher_norm(ts, student_metric, teacher_metric, mode):
    fig, ax = plt.subplots(figsize=(9, 5.2))
    for beta, group in ts.groupby('beta', sort=True):
        group = group.sort_values('step')
        line, = ax.plot(group['step'], group[student_metric], marker='o', linewidth=1.5, label=f'beta={beta:g}')
        ax.plot(group['step'], group[teacher_metric], linestyle='--', linewidth=1.3, color='red', alpha=0.75)
    ax.set_title(f'{mode}: {student_metric} with teacher reference at best final test-xent LR')
    ax.set_xlabel('SGD step')
    ax.set_ylabel(student_metric)
    ax.grid(True, alpha=0.25)
    ax.legend(ncols=2, fontsize='small')
    fig.tight_layout()
    fig.savefig(runs_dir / f'time_{mode}_{student_metric}_teacher_overlay.png', dpi=180)
    return fig

time_metrics = [
    'test_xent',
    'train_xent',
    'test_logit_mse',
    'train_logit_mse',
    'test_student_df_dh1_rms',
    'w1_cosine_fro',
]
norm_overlays = [
    ('test_student_h1_rms', 'test_teacher_h1_rms'),
    ('test_student_h2_rms', 'test_teacher_h2_rms'),
]


In [ ]:
for mode in modes:
    ts = best_lr_timeseries(df, best, mode)
    for metric in time_metrics:
        fig = plot_metric_over_time(ts, metric, mode)
        plt.show()
    for student_metric, teacher_metric in norm_overlays:
        fig = plot_student_teacher_norm(ts, student_metric, teacher_metric, mode)
        plt.show()
